<a href="https://colab.research.google.com/github/skumova/bist-100-teknik-analiz-ve-filtreleme-i-yi.ipynb/blob/claude%2Fkeen-pascal-W09xb/Simons_Quant_BIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jim Simons Kantitatif Strateji — BIST

**Temel Bileşenler:**
1. **HMM** — Gizli Markov Modeli ile piyasa rejimi tespiti (Bull/Bear/Sideways)
2. **İstatistiki Arbitraj** — Korelasyonlu çift hisseler, mean reversion (Z-score)
3. **Anomali Taraması** — Geçmiş veriyle istatistiksel edge tespiti
4. **Kelly Kriteri** — Pozisyon boyutlandırma
5. **%51 Kuralı** — Küçük ama tekrarlanabilir edge'leri birleştirme

> Not: Günlük veriyle çalışır (Colab/yfinance). Gerçek Rentec sistemi milisaniye verisi kullanır.

In [1]:
import subprocess, sys

def pip_install(pkg, label=None):
    label = label or pkg
    print(f"  {label} ...", end=" ")
    r = subprocess.run([sys.executable,"-m","pip","install","-q",pkg], capture_output=True, text=True)
    print("OK" if r.returncode==0 else f"HATA\n{r.stderr[-300:]}")

for pkg in [
    "pandas","numpy","requests","tqdm","openpyxl",
    "yfinance",
    "websocket-client",
    "websockets",
    "hmmlearn",
    "scikit-learn",
    "scipy",
    "statsmodels",
    "tradingview-screener",
]:
    pip_install(pkg)

# tvdatafeed -- GitHub (en guncel, BIST teknik veri icin en temiz kaynak)
pip_install(
    "git+https://github.com/rongardF/tvdatafeed.git",
    label="tvdatafeed (rongardF/GitHub)"
)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = "/content/drive/MyDrive/Simons_Quant"
    IN_COLAB = True
except Exception:
    ROOT = "/tmp/Simons_Quant"
    IN_COLAB = False

import os
os.makedirs(f"{ROOT}/cache", exist_ok=True)
os.makedirs(f"{ROOT}/raporlar", exist_ok=True)
print(f"\nRoot: {ROOT} | Colab: {IN_COLAB}")


  pandas ... OK
  numpy ... OK
  requests ... OK
  tqdm ... OK
  openpyxl ... OK
  yfinance ... OK
  websocket-client ... OK
  websockets ... OK
  hmmlearn ... OK
  scikit-learn ... OK
  scipy ... OK
  statsmodels ... OK
  tradingview-screener ... OK
  tvdatafeed (rongardF/GitHub) ... OK
Mounted at /content/drive

Root: /content/drive/MyDrive/Simons_Quant | Colab: True


In [2]:
import warnings, time, pickle
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm
warnings.filterwarnings("ignore")

# ── Strateji Parametreleri ────────────────────────────────────────────────────
N_BARS            = 500    # Fiyat geçmişi (gün)
HMM_N_STATES      = 3      # Piyasa rejimleri: Bull / Sideways / Bear
HMM_LOOKBACK      = 252    # HMM eğitim penceresi (1 yıl)
ZSCORE_WINDOW     = 20     # Mean reversion Z-score penceresi
ZSCORE_ENTRY      = 2.0    # Giriş eşiği (|z| > bu değer)
ZSCORE_EXIT       = 0.5    # Çıkış eşiği (|z| < bu değer)
CORR_MIN          = 0.70   # Çift seçimi min korelasyon
COINT_PVALUE      = 0.05   # Koentegrasyon p-value eşiği
KELLY_FRACTION    = 0.25   # Kelly fraksiyonu (0.25 = quarter Kelly)
MAX_POSITION_PCT  = 0.10   # Tek pozisyon maks %10
CACHE_TTL_H       = 6      # Cache geçerlilik (saat)

print("Konfigürasyon yüklendi.")


Konfigürasyon yüklendi.


In [3]:
# -- Tum BIST Hisse Listesi (459 hisse) --------------------------------
# Kaynak: mevcut liste + MCP BIST screener + tradingview-screener fallback
# Temel Analiz notebook ile eslestirilmis

BIST_SABIT = [
    "ACSEL","ADEL","ADESE","ADGYO","AEFES","AFYON","AGESA","AGHOL","AGROT","AHGAZ",
    "AKBNK","AKFGY","AKGRT","AKMGY","AKSA","AKSEN","AKSGY","AKSUE","AKTIF","ALARK",
    "ALBRK","ALCAR","ALFAS","ALGYO","ALKIM","ALKLC","ALMAD","ALTNY","ALVES","ANELE",
    "ANGEN","ANHYT","ANSGR","ARASE","ARCLK","ARDYZ","ARENA","ARSAN","ASELS","ASGYO",
    "ASTOR","ATAGY","ATAKP","ATATP","ATEKS","ATLAS","AVGYO","AVHOL","AVOD","AVPGY",
    "AYCES","AYGAZ","AZTEK","BAGFS","BAKAB","BALSU","BANVT","BARMA","BASCM","BASGZ",
    "BAYRK","BERA","BEYAZ","BFREN","BIMAS","BIOEN","BIZIM","BJKAS","BLCYT","BMEKS",
    "BMSTL","BNTAS","BORLS","BORSK","BOSSA","BRKO","BRKVY","BRMEN","BRSAN","BRYAT",
    "BSOKE","BTCIM","BUCIM","BURCE","BURVA","BVSAN","CANTE","CCOLA","CELHA","CEMAS",
    "CEMTS","CIMSA","CLEBI","CMBTN","CMENT","COGAS","COSMO","CRDFA","CRFSA","CUSAN",
    "CVKMD","CWENE","DAGHL","DAGI","DAPGM","DARDL","DENGE","DERHL","DESA","DESPC",
    "DEVA","DGATE","DGNMO","DITAS","DMSAS","DNISI","DOAS","DOBUR","DOCO","DOGUB",
    "DOHOL","DOSCH","DPAZR","DRDGE","DTRND","DURDO","DYOBY","DZGYO","ECILC","ECZYT",
    "EDIP","EFOR","EFORC","EGEEN","EGEPO","EGGUB","EGPRO","EGSER","EKGYO","EKIZ",
    "EKOS","EKSUN","ELITE","EMKEL","EMNIS","ENERY","ENJSA","ENKAI","ENSRI","EPLAS",
    "ERBOS","ERCB","EREGL","ESCAR","ESCOM","ESEN","ETGRI","ETYAT","EUHOL","EUPWR",
    "EUREN","EUYO","EVREN","FADE","FENER","FMIZP","FONET","FORMT","FORTE","FROTO",
    "FZLGY","GARAN","GARFA","GEDIK","GEDZA","GENIL","GENTS","GEREL","GESAN","GLRMK",
    "GLYHO","GMTAS","GNDUZ","GOLTS","GOODY","GOZDE","GRSEL","GRTHO","GSRAY","GUBRE",
    "GUBRF","GULFA","GUMER","GUNAY","GWIND","HALKB","HATEK","HDFGS","HEDEF","HEKTS",
    "HKTM","HLGYO","HTTBT","HUBVC","HUNER","HZNGY","ICBCT","IDGYO","IEDAS","IEYHO",
    "IHEVA","IHGZT","IHLAS","IHLGM","IHMAD","IHYAY","IMASM","INDES","INGRM","INTEM",
    "INVEO","IPEKE","IPMAT","ISCTR","ISFIN","ISGSY","ISGYO","ISMEN","ISYAT","ITTFK",
    "IZENR","JANTS","KAPLM","KAREL","KARSN","KARTN","KCHOL","KERVN","KFEIN","KGYO",
    "KLNMA","KLRHO","KMPUR","KNFRT","KOCMT","KONTR","KONYA","KOPOL","KORDS","KOZAA",
    "KOZAL","KRDMA","KRDMB","KRDMD","KRONT","KRPLS","KSTUR","KTLEV","KURTL","KUYAS",
    "KZBGY","LIDER","LIDFA","LINK","LKMNH","LOGO","LRSHO","LYKHO","MAALT","MACKO",
    "MAGEN","MAKIM","MANAS","MARKA","MARTI","MAVI","MEDTR","MEGAP","MEPET","MERCN",
    "MERIT","MERKO","METRO","METUR","MGROS","MIATK","MIGRS","MMCAS","MNDRS","MNDTR",
    "MOBTL","MOGAN","MPARK","MRGYO","MSGYO","MTRKS","MZHLD","NATEN","NETAS","NIBAS",
    "NIDDK","NTHOL","NTTUR","NUGYO","NUHCM","NXMGY","OBAMS","ODAS","OFSYM","OLMIP",
    "ONCSM","ONEN","ONRYT","ORGE","ORMA","OSTIM","OTKAR","OTTO","OYAKC","OYLUM",
    "OZGYO","OZKGY","OZRDN","OZSUB","PAGYO","PAHOL","PAMEL","PAPIL","PARSN","PASEU",
    "PATEK","PCILT","PEGYO","PEKGY","PENGD","PENTA","PETKM","PETUN","PGSUS","PINSU",
    "PKART","PKENT","PLTUR","PNSUT","POLHO","POLTK","PORTK","PRDGS","PRZMA","PSDTC",
    "PSGYO","QNBFB","QUAGR","RALYH","REEDR","RGYAS","RHEAG","RNPOL","RODRG","ROYAL",
    "RTALB","RUBNS","RYGYO","SAFKR","SAHOL","SANEL","SANFM","SANKO","SARKY","SASA",
    "SAYAS","SEKFK","SEKUR","SELEC","SELGD","SELVA","SEYKM","SILVR","SISE","SKBNK",
    "SKTAS","SMART","SMRTG","SNGYO","SOKM","SONME","SRVGY","SUMAS","SUNTK","SURGY",
    "SUWEN","SUZGT","TABGD","TARKM","TATGD","TAVHL","TBORG","TCELL","TDGYO","TEKTU",
    "TETMT","THYAO","TIRE","TKFEN","TKNSA","TLMAN","TMSN","TOASO","TRALT","TRCAS",
    "TRENJ","TRGYO","TRILC","TRMET","TRNSK","TSKB","TSPOR","TTKOM","TTRAK","TUCLK",
    "TUKAS","TUMAS","TUPRS","TUREX","TURGG","TURSG","UFUK","ULKER","ULUFA","ULUSE",
    "ULUUN","UNLU","USAK","USDAU","VAKBN","VAKFA","VAKFN","VANGD","VBTYZ","VERUS",
    "VESBE","VESTL","VKFYO","VKGYO","VRGYO","YATAS","YBTAS","YGGYO","YKBNK","YKGYO",
    "YKSLN","YONGA","YUNSA","ZEDUR","ZOREN","ZRGYO",
]

_TICKER_CACHE = Path(ROOT) / "cache" / "bist_tickers.txt"

def get_bist_symbols():
    # 0. Drive cache: onceki basarili screener sonucu varsa kullan
    if _TICKER_CACHE.exists():
        age_h = (time.time() - _TICKER_CACHE.stat().st_mtime) / 3600
        if age_h < 24:
            syms = _TICKER_CACHE.read_text().strip().split("\n")
            syms = [s for s in syms if s]
            if len(syms) > 100:
                print(f"Drive cache: {len(syms)} hisse (guncelleme: {age_h:.0f}s once)")
                return syms

    # 1. TradingView Screener (set_markets turkey) -- temel analiz ile ayni yontem
    try:
        from tradingview_screener import Query
        _, df = (
            Query()
            .set_markets("turkey")
            .select("name","close","volume","market_cap_basic")
            .limit(700)
            .get_scanner_data()
        )
        syms = df["ticker"].str.replace(r"^.*:", "", regex=True).tolist()
        if len(syms) > 100:
            _TICKER_CACHE.write_text("\n".join(sorted(syms)))
            print(f"TradingView Screener (turkey): {len(syms)} hisse -- Drive'a kaydedildi")
            return syms
    except Exception:
        pass

    # 2. BIST exchange sorgusu
    try:
        from tradingview_screener import Query, col as tvcol
        _, df = (
            Query()
            .select("name","close","volume")
            .where(tvcol("exchange").isin(["BIST"]), tvcol("type") == "stock")
            .limit(700)
            .get_scanner_data()
        )
        syms = df["name"].str.replace("BIST:","").tolist()
        if len(syms) > 100:
            _TICKER_CACHE.write_text("\n".join(sorted(syms)))
            print(f"TradingView Screener (BIST): {len(syms)} hisse")
            return syms
    except Exception:
        pass

    print(f"Sabit liste kullaniliyor: {len(BIST_SABIT)} hisse")
    return BIST_SABIT

SYMBOLS = get_bist_symbols()
print(f"Toplam: {len(SYMBOLS)} hisse taranacak")


TradingView Screener (turkey): 608 hisse -- Drive'a kaydedildi
Toplam: 608 hisse taranacak


In [4]:
# -- Fiyat Verisi: SADECE tvdatafeed (rongardF/GitHub) -----------------------
import logging
logging.getLogger("tvDatafeed.main").setLevel(logging.CRITICAL)

from tvDatafeed import TvDatafeed, Interval

# Tek seferlik baglanti — anonim (login gerekmez)
print("tvdatafeed baglaniyor...")
TV = TvDatafeed()

# Baglanti testi
_test = TV.get_hist("THYAO", "BIST", Interval.in_daily, n_bars=5)
if _test is not None and len(_test) > 0:
    print("tvdatafeed: BAGLI")
else:
    print("UYARI: Veri alinamadi, exchange/symbol kontrol edin")

def _cache_path(sym):
    return Path(ROOT) / "cache" / f"{sym}.pkl"

def get_price(sym: str, n: int = N_BARS) -> pd.DataFrame:
    cp = _cache_path(sym)
    if cp.exists() and (time.time() - cp.stat().st_mtime) / 3600 < CACHE_TTL_H:
        with open(cp, "rb") as f:
            return pickle.load(f)

    for attempt in range(3):
        try:
            df = TV.get_hist(
                symbol=sym,
                exchange="BIST",
                interval=Interval.in_daily,
                n_bars=n + 50,
            )
            if df is not None and len(df) > 30:
                df.columns = [c.lower() for c in df.columns]
                df.index = pd.to_datetime(df.index)
                df = df.tail(n).copy()
                df = df[df["close"] > 0].dropna(subset=["close"])
                if len(df) > 30:
                    with open(cp, "wb") as f:
                        pickle.dump(df, f)
                    return df
        except Exception:
            time.sleep(2 ** attempt)

    return pd.DataFrame()

# -- Benchmark: BIST-100 (TVC exchange) --------------------------------------
print("\nBenchmark (XU100) cekiliyor...")
BENCH = pd.DataFrame()
for attempt in range(3):
    try:
        _b = TV.get_hist("XU100", "TVC", Interval.in_daily, n_bars=700)
        if _b is not None and len(_b) > 100:
            _b.columns = [c.lower() for c in _b.columns]
            _b.index = pd.to_datetime(_b.index)
            BENCH = _b
            break
    except Exception:
        time.sleep(2 ** attempt)

if not BENCH.empty:
    BENCHMARK_CLOSE = BENCH["close"]
    print(f"Benchmark OK: {len(BENCH)} bar, son: {BENCH['close'].iloc[-1]:.0f}")
else:
    BENCHMARK_CLOSE = None
    print("UYARI: Benchmark alinamadi.")

# -- Test --------------------------------------------------------------------
print("\nTHYAO test...")
_t = get_price("THYAO")
if not _t.empty:
    print(f"OK -- {len(_t)} bar, son fiyat: {_t['close'].iloc[-1]:.2f} TL")
else:
    print("HATA: Veri alinamadi")


tvdatafeed baglaniyor...
tvdatafeed: BAGLI

Benchmark (XU100) cekiliyor...
UYARI: Benchmark alinamadi.

THYAO test...
OK -- 500 bar, son fiyat: 324.50 TL


In [5]:
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

def fit_hmm(close: pd.Series, n_states: int = HMM_N_STATES) -> dict:
    """
    Gizli Markov Modeli ile piyasa rejimini tespit eder.
    Özellikler: günlük getiri, volatilite, hacim değişimi
    Çıktı: her gün için rejim etiketi (0=Bear, 1=Sideways, 2=Bull)
    """
    if len(close) < HMM_LOOKBACK:
        return {"regime": None, "states": [], "probs": []}

    # Özellik matrisi
    ret      = close.pct_change().fillna(0)
    vol      = ret.rolling(5).std().fillna(0)
    ret_5d   = close.pct_change(5).fillna(0)

    X = np.column_stack([ret.values, vol.values, ret_5d.values])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # HMM eğitimi
    model = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=200,
        random_state=42,
    )
    try:
        model.fit(X_scaled[-HMM_LOOKBACK:])
        hidden_states = model.predict(X_scaled)
        state_probs   = model.predict_proba(X_scaled)
    except Exception as e:
        return {"regime": None, "states": [], "probs": []}

    # Rejim etiketleme: ortalama getiriye göre sırala
    state_returns = {}
    for s in range(n_states):
        mask = hidden_states == s
        state_returns[s] = ret.values[mask].mean() if mask.sum() > 0 else 0

    # Bull > Sideways > Bear sıralaması
    sorted_states = sorted(state_returns, key=lambda s: state_returns[s])
    label_map = {sorted_states[0]: "Bear", sorted_states[1]: "Sideways", sorted_states[2]: "Bull"}
    if n_states == 2:
        label_map = {sorted_states[0]: "Bear", sorted_states[1]: "Bull"}

    labeled = [label_map[s] for s in hidden_states]
    current_regime  = labeled[-1]
    current_probs   = {label_map[s]: float(state_probs[-1][s]) for s in range(n_states)}

    return {
        "regime":   current_regime,
        "prob":     current_probs,
        "history":  labeled,
        "last_5":   labeled[-5:],
    }

# Benchmark'a uygula
print("Piyasa rejimi analizi (BIST-100)...")
if not BENCH.empty:
    hmm_result = fit_hmm(BENCH["close"])
    print(f"  Güncel rejim  : {hmm_result['regime']}")
    print(f"  Son 5 gün     : {hmm_result['last_5']}")
    print(f"  Olasılıklar   : {hmm_result['prob']}")
    MARKET_REGIME = hmm_result["regime"]
else:
    MARKET_REGIME = "Unknown"
    print("Benchmark yok, rejim belirlenemedi.")


Piyasa rejimi analizi (BIST-100)...
Benchmark yok, rejim belirlenemedi.


In [6]:
from statsmodels.tsa.stattools import coint
from itertools import combinations

def find_cointegrated_pairs(symbols: list, price_dict: dict) -> list:
    """Koentegre cift hisseleri bulur (mean reversion icin ideal)."""
    valid = {s: price_dict[s]["close"] for s in symbols
             if s in price_dict and len(price_dict[s]) > 100}

    pairs = []
    syms  = list(valid.keys())
    total = len(list(combinations(syms, 2)))
    print(f"{len(syms)} hisse, {total} cift test ediliyor...")

    for s1, s2 in tqdm(combinations(syms, 2), total=total, desc="Cift tarama"):
        c1, c2 = valid[s1].align(valid[s2], join="inner")
        if len(c1) < 60:
            continue
        corr = c1.corr(c2)
        if abs(corr) < CORR_MIN:
            continue
        try:
            _, pval, _ = coint(c1.values, c2.values)
        except Exception:
            continue
        if pval < COINT_PVALUE:
            pairs.append({
                "s1": s1, "s2": s2,
                "corr": round(corr, 3),
                "coint_pval": round(pval, 4),
            })

    pairs.sort(key=lambda x: x["coint_pval"])
    print(f"\nBulunan koentegre cift: {len(pairs)}")
    return pairs


def calc_spread_zscore(c1: pd.Series, c2: pd.Series, window: int = ZSCORE_WINDOW) -> pd.Series:
    """Spread Z-score serisi."""
    c1, c2 = c1.align(c2, join="inner")
    ratio  = c1 / c2
    mean   = ratio.rolling(window).mean()
    std    = ratio.rolling(window).std()
    return (ratio - mean) / (std + 1e-9)


def get_pair_signal(s1: str, s2: str, price_dict: dict) -> dict:
    """Bir cift icin guncel mean reversion sinyali."""
    if s1 not in price_dict or s2 not in price_dict:
        return {}
    c1 = price_dict[s1]["close"]
    c2 = price_dict[s2]["close"]
    z  = calc_spread_zscore(c1, c2)
    if z.empty or pd.isna(z.iloc[-1]):
        return {}

    current_z = float(z.iloc[-1])
    if current_z > ZSCORE_ENTRY:
        action = f"LONG {s2} / SHORT {s1}"
    elif current_z < -ZSCORE_ENTRY:
        action = f"LONG {s1} / SHORT {s2}"
    elif abs(current_z) < ZSCORE_EXIT:
        action = "KAPAT (ortaya donus)"
    else:
        action = "Bekle"

    return {
        "s1": s1, "s2": s2,
        "zscore": round(current_z, 3),
        "action": action,
        "signal": abs(current_z) > ZSCORE_ENTRY,
    }

print("Pairs trading fonksiyonlari hazir.")


Pairs trading fonksiyonlari hazir.


In [7]:
from scipy import stats

def calc_edge_score(close: pd.Series, volume: pd.Series) -> dict:
    if len(close) < 60:
        return {"score": 0, "max_score": 6}

    close  = close.ffill().bfill()
    volume = volume.fillna(0)
    volume = volume.replace(0, volume[volume > 0].median() if (volume > 0).any() else 1)

    ret = close.pct_change().dropna()
    if len(ret) < 20:
        return {"score": 0, "max_score": 6}

    scores = {}

    # E1: Momentum — son 5G getirisi pozitif ve ortalamanin uzerinde
    try:
        ret_5d = close.pct_change(5).dropna()
        last_5d = float(ret_5d.iloc[-1])
        avg_5d  = float(ret_5d.mean())
        scores["E1_momentum"] = 1 if (last_5d > 0 and last_5d > avg_5d) else 0
        scores["ret_5d_pct"]  = round(last_5d * 100, 2)
    except Exception:
        scores["E1_momentum"] = 0

    # E2: Volatilite sikismasi — Bollinger bandi alt %35
    try:
        ma20  = close.rolling(20).mean()
        std20 = close.rolling(20).std()
        ratio = (std20 / (ma20 + 1e-9)).dropna()
        bb_pct = float(stats.percentileofscore(ratio.values, float(ratio.iloc[-1]))) / 100
        scores["bb_width_pct"] = round(bb_pct, 3)
        scores["E2_squeeze"]   = 1 if bb_pct < 0.35 else 0
    except Exception:
        scores["E2_squeeze"] = 0

    # E3: OBV yukseliyor
    try:
        obv   = (np.sign(close.diff()) * volume).cumsum()
        n     = min(20, len(obv))
        slope = float(np.polyfit(range(n), obv.iloc[-n:].values, 1)[0])
        scores["obv_slope"] = round(slope, 0)
        scores["E3_volume"] = 1 if slope > 0 else 0
    except Exception:
        scores["E3_volume"] = 0

    # E4: RSI momentum bolgesinde (45-75) ve yukselen trend
    try:
        delta = close.diff()
        gain  = delta.where(delta > 0, 0).rolling(14, min_periods=5).mean()
        loss  = (-delta).where(-delta > 0, 0).rolling(14, min_periods=5).mean()
        rsi   = 100 - 100 / (1 + gain / (loss + 1e-9))
        rsi_now  = float(rsi.iloc[-1])
        rsi_prev = float(rsi.iloc[-5])
        scores["rsi"] = round(rsi_now, 1)
        scores["E4_rsi_momentum"] = 1 if (45 < rsi_now < 75 and rsi_now > rsi_prev) else 0
    except Exception:
        scores["rsi"] = 50
        scores["E4_rsi_momentum"] = 0

    # E5: Fiyat 20G ortalamasinin uzerinde ve yukari kesisme
    try:
        ma20_val = float(close.rolling(20).mean().iloc[-1])
        ma20_prev = float(close.rolling(20).mean().iloc[-3])
        price_now = float(close.iloc[-1])
        price_prev = float(close.iloc[-3])
        # Fiyat MA'nin uzerinde VE son 3 gunde MA'yi asagi dan yukari kesiyor
        crossed_up = price_prev < ma20_prev and price_now > ma20_val
        above_ma   = price_now > ma20_val
        scores["E5_ma_cross"] = 1 if (above_ma and (crossed_up or price_now > ma20_val * 1.01)) else 0
    except Exception:
        scores["E5_ma_cross"] = 0

    # E6: Endekse gore guçlu (son 10G)
    try:
        if BENCHMARK_CLOSE is not None and len(BENCHMARK_CLOSE) > 10:
            bench_10d = float(BENCHMARK_CLOSE.pct_change(10).iloc[-1])
            stock_10d = float(close.pct_change(10).iloc[-1])
            scores["E6_rsc"] = 1 if stock_10d > bench_10d else 0
        else:
            scores["E6_rsc"] = 0
    except Exception:
        scores["E6_rsc"] = 0

    edge_keys = [k for k in scores if k.startswith("E")]
    scores["score"]     = sum(scores[k] for k in edge_keys)
    scores["max_score"] = len(edge_keys)
    return scores

print("Edge tarama fonksiyonu hazir.")
print("Kriterler: E1=Momentum | E2=Squeeze | E3=OBV | E4=RSI(45-75) | E5=MA20 | E6=RSC")


Edge tarama fonksiyonu hazir.
Kriterler: E1=Momentum | E2=Squeeze | E3=OBV | E4=RSI(45-75) | E5=MA20 | E6=RSC


In [8]:
def kelly_position(win_rate: float, avg_win: float, avg_loss: float,
                   portfolio_size: float = 1.0) -> dict:
    """
    Modifiye Kelly Kriteri ile optimal pozisyon boyutu.
    win_rate : kazanan işlem oranı (örn 0.52)
    avg_win  : ortalama kazanç (örn 0.03 = %3)
    avg_loss : ortalama kayıp (pozitif sayı, örn 0.02 = %2)
    """
    if avg_loss <= 0 or avg_win <= 0:
        return {"kelly_pct": 0, "position_size": 0}

    b = avg_win / avg_loss   # Kazanç/Kayıp oranı
    p = win_rate
    q = 1 - p

    kelly_full = (p * b - q) / b   # Tam Kelly
    kelly_frac = kelly_full * KELLY_FRACTION  # Quarter Kelly (daha güvenli)

    # Üst sınır
    kelly_frac = min(kelly_frac, MAX_POSITION_PCT)
    kelly_frac = max(kelly_frac, 0)

    return {
        "kelly_full_pct": round(kelly_full * 100, 2),
        "kelly_frac_pct": round(kelly_frac * 100, 2),
        "position_size":  round(kelly_frac * portfolio_size, 2),
        "win_rate":        win_rate,
        "payoff_ratio":    round(b, 2),
    }


def backtest_win_rate(close: pd.Series, entry_signal: pd.Series,
                      hold_days: int = 5) -> dict:
    """
    Geçmiş veriden win rate ve avg win/loss hesaplar.
    entry_signal: True olan günlerde alım yapıldığını varsayar.
    """
    wins, losses = [], []
    signal_dates = entry_signal[entry_signal].index

    for dt in signal_dates:
        try:
            idx = close.index.get_loc(dt)
            if idx + hold_days >= len(close):
                continue
            entry = close.iloc[idx]
            exit_ = close.iloc[idx + hold_days]
            ret   = (exit_ - entry) / entry
            if ret > 0:
                wins.append(ret)
            else:
                losses.append(abs(ret))
        except Exception:
            continue

    if not wins and not losses:
        return {"win_rate": 0.5, "avg_win": 0.02, "avg_loss": 0.02, "n_trades": 0}

    total  = len(wins) + len(losses)
    return {
        "win_rate": round(len(wins)/total, 3) if total > 0 else 0.5,
        "avg_win":  round(np.mean(wins), 4)  if wins   else 0.02,
        "avg_loss": round(np.mean(losses),4) if losses else 0.02,
        "n_trades": total,
    }

print("Kelly pozisyon boyutlandırma hazır.")


Kelly pozisyon boyutlandırma hazır.


In [9]:
# -- Tum Hisselerin Verisini Indir (tvdatafeed) ------------------------------
print(f"Veri indirme basliyor: {len(SYMBOLS)} hisse (tvdatafeed)")
print("-" * 50)

PRICE_DATA = {}
failed = []

for sym in tqdm(SYMBOLS, desc="Indiriliyor"):
    df = get_price(sym)
    if not df.empty:
        PRICE_DATA[sym] = df
    else:
        failed.append(sym)
    time.sleep(0.1)  # Rate limit

print(f"\nBasarili : {len(PRICE_DATA)}/{len(SYMBOLS)}")
print(f"Basarisiz: {len(failed)}")
if failed:
    print(f"  {failed[:20]}")


Veri indirme basliyor: 608 hisse (tvdatafeed)
--------------------------------------------------


Indiriliyor: 100%|██████████| 608/608 [08:23<00:00,  1.21it/s]


Basarili : 600/608
Basarisiz: 8
  ['ISVEA', 'GOLDA', 'SOHOE', 'BETAE', 'SSAAT', 'EKIM', 'SARAE', 'ORZAX']


In [10]:
# -- Pattern Madenciligi: gecmis buyuk yukselisleri ogren ---------------------
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

MIN_RISE_PCT   = 0.30   # Dusuruldu: %30 (BIST icin daha gercekci)
RISE_WINDOW    = 60     # Arttirildi: 60 gunde bu yukselis
PRE_RISE_DAYS  = 20     # Yukselis oncesi feature penceresi
MIN_SIMILARITY = 0.65   # Benzerlik esigi (biraz dusuk)


def extract_features(close, volume, start_idx):
    end_idx = start_idx + PRE_RISE_DAYS
    if end_idx > len(close):
        return None
    c = close.iloc[start_idx:end_idx].copy()
    v = volume.iloc[start_idx:end_idx].copy()

    # NaN / sifir temizligi
    c = c.fillna(method="ffill").fillna(method="bfill")
    v = v.fillna(0).replace(0, 1)  # Sifir hacmi 1'e esitle

    if c.std() < 1e-9 or len(c) < PRE_RISE_DAYS:
        return None

    c_mean = c.mean()
    if c_mean == 0:
        return None

    ret   = c.pct_change().fillna(0)
    v_rel = v / (v.mean() + 1e-9)
    obv   = (np.sign(c.diff().fillna(0)) * v).cumsum()

    delta = c.diff().fillna(0)
    gain  = delta.where(delta > 0, 0).rolling(5, min_periods=1).mean()
    loss  = (-delta).where(-delta > 0, 0).rolling(5, min_periods=1).mean()
    rs    = float(gain.iloc[-1]) / (float(loss.iloc[-1]) + 1e-9)
    rsi_n = (100 - 100 / (1 + rs)) / 100

    try:
        vol_slope = float(np.polyfit(range(PRE_RISE_DAYS), v_rel.values, 1)[0])
        obv_slope = float(np.polyfit(range(PRE_RISE_DAYS), obv.values, 1)[0])
        short_vol = float(c.rolling(5, min_periods=1).std().iloc[-1])
        long_vol  = float(c.rolling(PRE_RISE_DAYS, min_periods=5).std().iloc[-1])
        vol_ratio = short_vol / (long_vol + 1e-9)
    except Exception:
        return None

    price_range = (float(c.max()) - float(c.min())) / (c_mean + 1e-9)
    band_pos    = (float(c.iloc[-1]) - float(c.min())) / (float(c.max()) - float(c.min()) + 1e-9)
    c_slope     = float(c.iloc[-5:].mean()) / (float(c.iloc[:5].mean()) + 1e-9) - 1
    mom_diff    = float(ret.iloc[-5:].mean()) - float(ret.iloc[:5].mean())

    feat = np.array([
        float(ret.mean()), float(ret.std()), price_range, band_pos,
        c_slope, mom_diff, float(v_rel.mean()), vol_slope,
        float(v_rel.iloc[-5:].mean() / (v_rel.iloc[:5].mean() + 1e-9)),
        obv_slope, rsi_n, vol_ratio,
    ])

    return feat if not np.any(np.isnan(feat)) and not np.any(np.isinf(feat)) else None


def mine_patterns(price_dict):
    all_features, all_labels = [], []
    skipped_short = skipped_feat = found = 0

    print(f"Pattern madenciligi: {len(price_dict)} hisse")
    print(f"Kriter: {RISE_WINDOW} gunde >%{int(MIN_RISE_PCT*100)} yukselis")

    for sym, df in tqdm(price_dict.items(), desc="Pattern tarama"):
        close  = df["close"].reset_index(drop=True)
        volume = (df["volume"].reset_index(drop=True)
                  if "volume" in df.columns
                  else pd.Series(np.ones(len(close))))
        volume = volume.fillna(1).replace(0, 1)
        dates  = df.index

        min_len = PRE_RISE_DAYS + RISE_WINDOW + 5
        if len(close) < min_len:
            skipped_short += 1
            continue

        for i in range(PRE_RISE_DAYS, len(close) - RISE_WINDOW):
            entry = float(close.iloc[i])
            if entry <= 0 or np.isnan(entry):
                continue
            future_max = float(close.iloc[i: i + RISE_WINDOW].max())
            rise = (future_max - entry) / (entry + 1e-9)
            if rise < MIN_RISE_PCT:
                continue
            feat = extract_features(close, volume, i - PRE_RISE_DAYS)
            if feat is None:
                skipped_feat += 1
                continue
            all_features.append(feat)
            all_labels.append({
                "sym":      sym,
                "rise_pct": round(rise * 100, 1),
                "date":     str(dates[i]) if i < len(dates) else "?",
            })
            found += 1

    print(f"  Kisa veri atlanan : {skipped_short}")
    print(f"  Feature hatasi    : {skipped_feat}")
    print(f"  Bulunan pattern   : {found}")

    if not all_features:
        print("UYARI: Pattern bulunamadi!")
        return np.array([]), [], None

    matrix = np.array(all_features)
    scaler = StandardScaler()
    matrix_scaled = scaler.fit_transform(matrix)
    rises = [l["rise_pct"] for l in all_labels]
    print(f"Ort. yukselis: %{np.mean(rises):.1f}  |  Max: %{max(rises):.1f}")
    return matrix_scaled, all_labels, scaler


PATTERN_MATRIX, PATTERN_LABELS, PATTERN_SCALER = mine_patterns(PRICE_DATA)
print(f"\nPattern kutuphanesi: {len(PATTERN_LABELS)} pattern")


Pattern madenciligi: 600 hisse
Kriter: 60 gunde >%30 yukselis


Pattern tarama: 100%|██████████| 600/600 [04:54<00:00,  2.04it/s]

  Kisa veri atlanan : 3
  Feature hatasi    : 0
  Bulunan pattern   : 69572
Ort. yukselis: %65.8  |  Max: %987.4

Pattern kutuphanesi: 69572 pattern


In [11]:
# -- Guncel hisseleri pattern kutuphanesiyle karsilastir ----------------------

def scan_by_similarity(price_dict, pattern_matrix, pattern_labels, scaler):
    if len(pattern_matrix) == 0:
        print("Pattern kutuphanesi bos!")
        return []
    results = []
    print(f"Benzerlik taramasi: {len(price_dict)} hisse...")
    for sym, df in tqdm(price_dict.items(), desc="Benzerlik tarama"):
        close  = df["close"].reset_index(drop=True)
        volume = (df["volume"].reset_index(drop=True)
                  if "volume" in df.columns
                  else pd.Series(np.ones(len(close))))
        if len(close) < PRE_RISE_DAYS + 5:
            continue
        feat = extract_features(close, volume, len(close) - PRE_RISE_DAYS)
        if feat is None:
            continue
        feat_scaled     = scaler.transform(feat.reshape(1, -1))
        sims            = cosine_similarity(feat_scaled, pattern_matrix)[0]
        top_idx         = np.argsort(sims)[-10:][::-1]
        top_sim         = float(sims[top_idx[0]])
        avg_top5        = float(sims[top_idx[:5]].mean())
        if avg_top5 < MIN_SIMILARITY:
            continue
        top_patterns    = [pattern_labels[i] for i in top_idx[:10]]
        avg_rise        = float(np.mean([p["rise_pct"] for p in top_patterns]))
        max_rise        = float(max(p["rise_pct"] for p in top_patterns))
        benzer          = ", ".join(list(set(p["sym"] for p in top_patterns[:5])))
        results.append({
            "hisse":            sym,
            "fiyat":            round(float(df["close"].iloc[-1]), 2),
            "benzerlik_skoru":  round(avg_top5, 3),
            "max_benzerlik":    round(top_sim, 3),
            "tahmini_yukselis": round(avg_rise, 1),
            "max_yukselis":     round(max_rise, 1),
            "benzer_hisseler":  benzer,
        })
    results.sort(key=lambda x: (x["benzerlik_skoru"], x["tahmini_yukselis"]), reverse=True)
    return results


SIMILARITY_RESULTS = scan_by_similarity(
    PRICE_DATA, PATTERN_MATRIX, PATTERN_LABELS, PATTERN_SCALER
)
print(f"\nBenzer pattern bulunan hisse: {len(SIMILARITY_RESULTS)}")
if SIMILARITY_RESULTS:
    df_sim = pd.DataFrame(SIMILARITY_RESULTS)
    print("\n-- EN YUKSEK BENZERLIK SKORLARI -----------------------------------")
    print(df_sim[["hisse","fiyat","benzerlik_skoru","tahmini_yukselis",
                  "max_yukselis","benzer_hisseler"]].head(20).to_string(index=False))


Benzerlik taramasi: 600 hisse...


Benzerlik tarama: 100%|██████████| 600/600 [00:14<00:00, 42.29it/s]


Benzer pattern bulunan hisse: 600

-- EN YUKSEK BENZERLIK SKORLARI -----------------------------------
hisse   fiyat  benzerlik_skoru  tahmini_yukselis  max_yukselis                   benzer_hisseler
ALTIN   66.33            0.996              71.5         210.7 OZYSR, TGSAS, CVKMD, QNBFK, YUNSA
RAYSG  171.00            0.995              42.4          73.3  HUNER, BUCIM, SEGYO, MEGMT, ADEL
PGSUS  163.70            0.994             138.4         717.6 CUSAN, EDATA, TRHOL, AKSUE, DIRIT
KARTN  221.10            0.994              85.7         158.4   TERA, DERHL, AYES, KLNMA, ALKLC
OFSYM   51.95            0.994              78.3         192.6 ONCSM, MOPAS, ALKIM, GARFA, YONGA
BIENY   20.92            0.994              75.6         122.6 SAFKR, GARFA, MOBTL, AKSUE, FMIZP
DOKTA   23.18            0.994              59.2         109.6  AVOD, ADGYO, MAKTK, MOGAN, AKSUE
RUBNS   21.74            0.994              57.1          90.3               ALFAS, MOPAS, KTSKR
BFREN  129.40          

In [12]:
# ── FILTRE A: Tamamlanmis Yukselis Elemesi ────────────────────────────────────
# Üç koşul birden varsa hisse "tamamlandi" sayilir ve listeden cikarilir:
#   1. Son 40 günde zaten >%25 yükseliş yaşanmış
#   2. Son 5 günlük fiyat trendi negatif (momentum kırıldı)
#   3. Hacim son 5 günde ortalama altında (alıcı çekildi)

def is_rise_complete(close: pd.Series, volume: pd.Series) -> dict:
    if len(close) < 45:
        return {"complete": False, "reason": "veri_yetersiz", "rise_40d_pct": 0, "momentum": "bilinmiyor", "hacim_durumu": "bilinmiyor", "vol_oran": 0}

    # Kosul 1: Son 40 gunde yükselis
    rise_40d = (float(close.iloc[-1]) / float(close.iloc[-40]) - 1)

    # Kosul 2: Son 5 gun momentum
    slope_5d = float(np.polyfit(range(5), close.iloc[-5:].values, 1)[0])
    momentum_negative = slope_5d < 0

    # Kosul 3: Hacim kuruyor
    vol_avg = float(volume.iloc[-40:].mean())
    vol_now = float(volume.iloc[-5:].mean())
    volume_drying = vol_now < vol_avg * 0.75

    complete = (rise_40d > 0.25) and momentum_negative and volume_drying

    return {
        "complete":         complete,
        "rise_40d_pct":     round(rise_40d * 100, 1),
        "momentum":         "negatif" if momentum_negative else "pozitif",
        "hacim_durumu":     "kuruyor" if volume_drying else "normal",
        "vol_oran":         round(vol_now / (vol_avg + 1e-9), 2),
    }


# ── FILTRE B: Sessiz Birikim Tespiti ─────────────────────────────────────────
# Tahta yapıcı henüz pozisyon alıyor:
#   1. Fiyat dar bantta sıkışmış (volatilite düşük)
#   2. OBV yükseliyor (içeriden alım var)
#   3. Hacim artış trendinde ama fiyat hareket etmemiş
#   4. Yüksek hacimli günlerde fiyat artıyor, düşük hacimli günlerde düşüyor

def detect_accumulation(close: pd.Series, volume: pd.Series,
                        window: int = 20) -> dict:
    if len(close) < window + 5:
        return {"accumulating": False, "score": 0}

    c = close.iloc[-window:].copy()
    v = volume.iloc[-window:].copy()
    v = v.replace(0, 1)

    # B1: Dar bant — fiyat bandı %12'den dar
    band_pct = (float(c.max()) - float(c.min())) / (float(c.mean()) + 1e-9)
    b1 = band_pct < 0.12

    # B2: OBV yukseliyor
    obv   = (np.sign(c.diff().fillna(0)) * v).cumsum()
    obv_slope = float(np.polyfit(range(window), obv.values, 1)[0])
    b2 = obv_slope > 0

    # B3: Hacim artis trendinde (giderek büyüyor)
    vol_slope = float(np.polyfit(range(window), v.values, 1)[0])
    b3 = vol_slope > 0

    # B4: Hacim/fiyat uyumu — yüksek hacimli günlerde fiyat artiyor
    vol_median = v.median()
    ret = c.pct_change().fillna(0)
    high_vol_ret = float(ret[v > vol_median].mean()) if (v > vol_median).sum() > 2 else 0
    low_vol_ret  = float(ret[v <= vol_median].mean()) if (v <= vol_median).sum() > 2 else 0
    b4 = high_vol_ret > low_vol_ret

    # B5: Fiyat son 5 gunde yukselis baslamadi (henuz erken)
    recent_rise = float(c.iloc[-1]) / float(c.iloc[-6]) - 1 if len(c) >= 6 else 0
    b5 = recent_rise < 0.08  # Henuz %8'den az hareket

    score = sum([b1, b2, b3, b4, b5])

    return {
        "accumulating":  score >= 3,
        "score":         score,
        "band_pct":      round(band_pct * 100, 1),
        "obv_slope":     "artiyor" if b2 else "dusuyor",
        "hacim_trendi":  "artiyor" if b3 else "dusuyor",
        "smart_money":   "uyumlu" if b4 else "uyumsuz",
        "erken_mi":      "evet" if b5 else "hareket_basladi",
        "b1_dar_bant":   b1,
        "b2_obv":        b2,
        "b3_hacim":      b3,
        "b4_smart":      b4,
        "b5_erken":      b5,
    }


# ── Similarity sonuçlarını filtrele ve zenginleştir ───────────────────────────
print("Filtreler uygulanıyor...")
print(f"  Girdi: {len(SIMILARITY_RESULTS)} hisse")

FILTERED_RESULTS = []
stats = {"tamamlandi": 0, "birikim_yok": 0, "gecti": 0}

for r in SIMILARITY_RESULTS:
    sym = r["hisse"]
    df  = PRICE_DATA.get(sym)
    if df is None or df.empty:
        continue

    close  = df["close"]
    volume = df["volume"] if "volume" in df.columns else pd.Series(np.ones(len(close)), index=close.index)
    volume = volume.fillna(1).replace(0, 1)

    # --- Filtre A: tamamlandi mi? ---
    fa = is_rise_complete(close, volume)
    if fa["complete"]:
        stats["tamamlandi"] += 1
        continue  # Listeden cikart

    # --- Filtre B: birikim var mi? ---
    fb = detect_accumulation(close, volume)

    # Birikim skoru 0 olan hisseleri de dahil et ama skorla isaretlensin
    result = {
        **r,
        # Filtre A bilgisi
        "yukselis_40d_%":   fa["rise_40d_pct"],
        "momentum":         fa["momentum"],
        # Filtre B bilgisi
        "birikim_skoru":    f"{fb['score']}/5",
        "birikim":          "EVET" if fb["accumulating"] else "hayir",
        "band_%":           fb["band_pct"],
        "obv":              fb["obv_slope"],
        "smart_money":      fb["smart_money"],
        "erken_mi":         fb["erken_mi"],
    }
    FILTERED_RESULTS.append(result)
    stats["gecti"] += 1

# Sirala: once birikim olanlar, sonra benzerlik skoru
FILTERED_RESULTS.sort(key=lambda x: (
    1 if x["birikim"] == "EVET" else 0,
    x["benzerlik_skoru"]
), reverse=True)

print(f"  Tamamlandi (elendi)  : {stats['tamamlandi']}")
print(f"  Filtre gecen         : {stats['gecti']}")
print(f"  Birikim tespit edilen: {sum(1 for r in FILTERED_RESULTS if r['birikim']=='EVET')}")

if FILTERED_RESULTS:
    df_filtered = pd.DataFrame(FILTERED_RESULTS)
    print("\n── FILTRELI SONUCLAR (Birikim Olanlar Once) ─────────────────")
    cols = ["hisse","fiyat","benzerlik_skoru","birikim","birikim_skoru",
            "band_%","obv","smart_money","erken_mi",
            "yukselis_40d_%","tahmini_yukselis"]
    print(df_filtered[cols].head(25).to_string(index=False))
else:
    print("Filtre sonrasi hisse kalmadi. Esikleri gozden gecir.")


Filtreler uygulanıyor...
  Girdi: 600 hisse
  Tamamlandi (elendi)  : 10
  Filtre gecen         : 590
  Birikim tespit edilen: 295

── FILTRELI SONUCLAR (Birikim Olanlar Once) ─────────────────
hisse   fiyat  benzerlik_skoru birikim birikim_skoru  band_%     obv smart_money        erken_mi  yukselis_40d_%  tahmini_yukselis
ALTIN   66.33            0.996    EVET           3/5    10.9 dusuyor      uyumlu            evet           -14.4              71.5
PGSUS  163.70            0.994    EVET           4/5    10.9 dusuyor      uyumlu            evet            -5.6             138.4
KARTN  221.10            0.994    EVET           3/5    50.5 artiyor      uyumlu hareket_basladi           100.5              85.7
BFREN  129.40            0.994    EVET           3/5     7.6 dusuyor      uyumlu            evet            -9.0              57.0
KAYSE    4.05            0.994    EVET           3/5     9.6 dusuyor     uyumsuz            evet           -13.1              48.9
PETKM   21.96        

In [13]:
# -- Edge Tarama --------------------------------------------------------------
print(f"Edge taramasi basliyor... Piyasa rejimi: {MARKET_REGIME}")
print(f"Esik: minimum 2/6 sinyal")
print("-" * 55)

edge_results = []
score_dist   = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}

for sym, df in tqdm(PRICE_DATA.items(), desc="Edge tarama"):
    try:
        close  = df["close"]
        volume = (df["volume"] if "volume" in df.columns
                  else pd.Series(np.ones(len(close)), index=close.index))
        if volume.empty:
            volume = pd.Series(np.ones(len(close)), index=close.index)

        sc = calc_edge_score(close, volume)
        score_dist[min(sc["score"], 6)] += 1

        if sc["score"] < 2:
            continue

        hmm         = fit_hmm(close, n_states=2)
        stock_regime = hmm.get("regime", "?")

        delta = close.diff()
        gain  = delta.where(delta>0,0).rolling(14, min_periods=5).mean()
        loss  = (-delta).where(-delta<0,0).rolling(14, min_periods=5).mean()
        rsi_s = 100 - 100/(1 + gain/(loss+1e-9))
        obv_s = (np.sign(close.diff()) * volume).cumsum()
        obv_slope_s = obv_s.rolling(10).apply(
            lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True)
        signal_series = (rsi_s < 60) & (obv_slope_s > 0)

        bt    = backtest_win_rate(close.iloc[-252:], signal_series.iloc[-252:], hold_days=5)
        kelly = kelly_position(bt["win_rate"], bt["avg_win"], bt["avg_loss"])

        edge_results.append({
            "hisse":        sym,
            "fiyat":        round(float(close.iloc[-1]), 2),
            "skor":         f"{sc['score']}/{sc['max_score']}",
            "RSI":          sc.get("rsi", "-"),
            "ret5d_%":      sc.get("ret_5d_pct", "-"),
            "E1_mom":       "v" if sc.get("E1_momentum") else ".",
            "E2_sqz":       "v" if sc.get("E2_squeeze")  else ".",
            "E3_obv":       "v" if sc.get("E3_volume")   else ".",
            "E4_rsi":       "v" if sc.get("E4_rsi_momentum") else ".",
            "E5_ma":        "v" if sc.get("E5_ma_cross") else ".",
            "E6_rsc":       "v" if sc.get("E6_rsc")      else ".",
            "rejim":        stock_regime,
            "win_%":        round(bt["win_rate"]*100, 1),
            "n_islem":      bt["n_trades"],
            "kelly_%":      kelly["kelly_frac_pct"],
        })
    except Exception as e:
        continue

print(f"\nSkor dagilimi: {score_dist}")
print(f"Edge sinyali (>=2): {len(edge_results)} hisse")


Edge taramasi basliyor... Piyasa rejimi: Unknown
Esik: minimum 2/6 sinyal
-------------------------------------------------------


Edge tarama: 100%|██████████| 600/600 [00:25<00:00, 23.54it/s]


Skor dagilimi: {0: 189, 1: 159, 2: 83, 3: 103, 4: 66, 5: 0, 6: 0}
Edge sinyali (>=2): 6 hisse


In [14]:
# ── Koentegre Çift Tarama (Mean Reversion) ───────────────────────────────────
# Sadece en likit 50 hisseyi çift testine sok (kombinasyon sayısı makul kalsın)
liquid_syms = list(PRICE_DATA.keys())[:50]
print(f"Çift taraması: {liquid_syms[:50]} hissede...")

PAIRS = find_cointegrated_pairs(liquid_syms, PRICE_DATA)

# Aktif çift sinyalleri
pair_signals = []
for p in PAIRS[:30]:  # En iyi 30 çift
    sig = get_pair_signal(p["s1"], p["s2"], PRICE_DATA)
    if sig.get("signal"):
        pair_signals.append({**p, **sig})

print(f"Aktif çift sinyali: {len(pair_signals)}")
if pair_signals:
    df_pairs = pd.DataFrame(pair_signals)
    print(df_pairs[["s1","s2","corr","coint_pval","zscore","action"]].to_string(index=False))


Çift taraması: ['ASELS', 'QNBTR', 'DSTKF', 'HEDEF', 'TUPRS', 'GARAN', 'ENKAI', 'KCHOL', 'BIMAS', 'THYAO', 'KTLEV', 'AKBNK', 'VAKBN', 'FROTO', 'EREGL', 'ASTOR', 'YKBNK', 'HALKB', 'CCOLA', 'TCELL', 'ODINE', 'TTKOM', 'OZATD', 'SAHOL', 'ISDMR', 'SELEC', 'TRALT', 'TOASO', 'GUBRF', 'KLRHO', 'TURSG', 'SISE', 'AKSEN', 'QNBFK', 'SASA', 'INVES', 'AEFES', 'ENJSA', 'KENT', 'MAGEN', 'MGROS', 'TERA', 'TRGYO', 'OYAKC', 'TAVHL', 'KLNMA', 'LIDER', 'AHGAZ', 'PGSUS', 'ENERY'] hissede...
50 hisse, 1225 cift test ediliyor...


Cift tarama: 100%|██████████| 1225/1225 [00:11<00:00, 107.97it/s]



Bulunan koentegre cift: 98
Aktif çift sinyali: 4
   s1    s2   corr  coint_pval  zscore                   action
DSTKF ASTOR  0.906      0.0001  -2.353 LONG DSTKF / SHORT ASTOR
KCHOL  SISE  0.903      0.0002   2.015  LONG SISE / SHORT KCHOL
DSTKF LIDER  0.833      0.0023  -2.394 LONG DSTKF / SHORT LIDER
BIMAS PGSUS -0.775      0.0026   2.015 LONG PGSUS / SHORT BIMAS


In [15]:
# -- Nihai Rapor --------------------------------------------------------------
now_str = datetime.now().strftime("%d.%m.%Y %H:%M")
sep = "=" * 65
print(f"\n{sep}")
print(f"  JIM SIMONS KANTITATIF TARAMA -- {now_str}")
print(f"{sep}")
print(f"  Piyasa rejimi       : {MARKET_REGIME}")
print(f"  Taranan hisse       : {len(PRICE_DATA)}")
print(f"  Pattern kutuphanesi : {len(PATTERN_LABELS)} gecmis pattern")
print(f"  Benzerlik sinyali   : {len(SIMILARITY_RESULTS)}")
print(f"  Edge sinyali        : {len(edge_results)}")
print(f"  Aktif cift          : {len(pair_signals)}")
print(f"{sep}\n")

ts = datetime.now().strftime("%Y%m%d_%H%M")
xl = f"{ROOT}/raporlar/simons_quant_{ts}.xlsx"

with pd.ExcelWriter(xl, engine="openpyxl") as writer:
    if FILTERED_RESULTS:
        df_sim = pd.DataFrame(FILTERED_RESULTS).sort_values('benzerlik_skoru', ascending=False)
        df_sim.to_excel(writer, sheet_name='Filtreli_Adaylar', index=False)
        print('-- FILTRELI ADAYLAR (A+B Filtre) ----------------------')
        print(df_sim[['hisse','fiyat','benzerlik_skoru','birikim','birikim_skoru',
                      'tahmini_yukselis','smart_money','erken_mi']].head(20).to_string(index=False))
    if SIMILARITY_RESULTS:
        df_sim = pd.DataFrame(SIMILARITY_RESULTS).sort_values("benzerlik_skoru", ascending=False)
        df_sim.to_excel(writer, sheet_name="Pattern_Benzerligi", index=False)
        print("-- PATTERN BENZERLIGI (Tahta Yapici Izi) ----------------------")
        print(df_sim[["hisse","fiyat","benzerlik_skoru","tahmini_yukselis",
                       "max_yukselis","benzer_hisseler"]].head(20).to_string(index=False))
    if edge_results:
        df_edge = pd.DataFrame(edge_results).sort_values(
            ["skor","win_%"], ascending=False)
        df_edge.to_excel(writer, sheet_name="Edge_Sinyaller", index=False)
        print("\n-- EDGE SINYALLERI --------------------------------------------")
        print(df_edge.head(20).to_string(index=False))
    if pair_signals:
        pd.DataFrame(pair_signals).to_excel(writer, sheet_name="Cift_Islemler", index=False)
    if SIMILARITY_RESULTS and edge_results:
        sim_set  = {r["hisse"] for r in SIMILARITY_RESULTS}
        edge_set = {r["hisse"] for r in edge_results}
        kesisim  = sim_set & edge_set
        if kesisim:
            star = "*" * 55
            print(f"\n{star}")
            print(f"  KESISIM: Her iki yontemde sinyal veren hisseler:")
            print(f"  {', '.join(sorted(kesisim))}")
            print(f"{star}")
            df_sim[df_sim["hisse"].isin(kesisim)].to_excel(
                writer, sheet_name="Kesisim_Guclu", index=False)

print(f"\nExcel: {xl}")



  JIM SIMONS KANTITATIF TARAMA -- 20.07.2026 10:22
  Piyasa rejimi       : Unknown
  Taranan hisse       : 600
  Pattern kutuphanesi : 69572 gecmis pattern
  Benzerlik sinyali   : 600
  Edge sinyali        : 6
  Aktif cift          : 4

-- FILTRELI ADAYLAR (A+B Filtre) ----------------------
hisse   fiyat  benzerlik_skoru birikim birikim_skoru  tahmini_yukselis smart_money        erken_mi
ALTIN   66.33            0.996    EVET           3/5              71.5      uyumlu            evet
RAYSG  171.00            0.995   hayir           1/5              42.4     uyumsuz            evet
GRNYO   15.85            0.994   hayir           1/5              40.5     uyumsuz            evet
TKNSA   18.32            0.994   hayir           1/5              41.1     uyumsuz            evet
MHRGY    3.57            0.994   hayir           1/5              46.2     uyumsuz            evet
RUBNS   21.74            0.994   hayir           1/5              57.1     uyumsuz            evet
OZKGY   13.52

In [16]:
# ── Tekil Hisse Detay Analizi ─────────────────────────────────────────────────
def full_analysis(sym: str, portfolio_tl: float = 100_000):
    df = PRICE_DATA.get(sym, pd.DataFrame())
    if df.empty:
        df = get_price(sym)
    if df is None or df.empty:
        print(f"{sym}: veri yok")
        return

    close  = df["close"]
    volume = df.get("volume", pd.Series(np.ones(len(close)), index=close.index))

    print(f"\n{'='*55}")
    print(f"  {sym} — SIMONS KANTİTATİF ANALİZ")
    print(f"{'='*55}")
    print(f"  Fiyat      : {close.iloc[-1]:.2f} TL")
    print(f"  1G         : {(close.iloc[-1]/close.iloc[-2]-1)*100:+.2f}%")
    print(f"  5G         : {(close.iloc[-1]/close.iloc[-5]-1)*100:+.2f}%")
    print(f"  20G        : {(close.iloc[-1]/close.iloc[-20]-1)*100:+.2f}%")

    print("\n── HMM Rejim Analizi ──────────────────────────")
    hmm = fit_hmm(close)
    print(f"  Güncel rejim : {hmm['regime']}")
    print(f"  Son 5 gün    : {hmm['last_5']}")
    probs = hmm.get("prob", {})
    for k, v in probs.items():
        print(f"  {k:10}: %{v*100:.1f}")

    print("\n── Edge Skorları ──────────────────────────────")
    sc = calc_edge_score(close, volume)
    for k, v in sc.items():
        if k.startswith("E"):
            print(f"  {k}: {'✓' if v else '✗'}")
    print(f"  TOPLAM: {sc['score']}/{sc['max_score']}")

    print("\n── Kelly Pozisyon ─────────────────────────────")
    delta = close.diff()
    gain  = delta.where(delta>0,0).rolling(14).mean()
    loss  = (-delta).where(delta<0,0).rolling(14).mean()
    rsi_s = 100 - 100/(1 + gain/(loss+1e-9))
    obv_s = (np.sign(close.diff()) * volume).cumsum()
    obv_slope_s = obv_s.rolling(10).apply(lambda x: np.polyfit(range(len(x)),x,1)[0], raw=True)
    sig = (rsi_s < 48) & (obv_slope_s > 0)
    bt  = backtest_win_rate(close, sig, hold_days=5)
    k   = kelly_position(bt["win_rate"], bt["avg_win"], bt["avg_loss"], portfolio_tl)
    print(f"  Geçmiş işlem : {bt['n_trades']} adet")
    print(f"  Win rate     : %{bt['win_rate']*100:.1f}")
    print(f"  Payoff oranı : {k['payoff_ratio']:.2f}x")
    print(f"  Tam Kelly    : %{k['kelly_full_pct']:.2f}")
    print(f"  Quarter Kelly: %{k['kelly_frac_pct']:.2f}")
    print(f"  Pozisyon TL  : {k['position_size']:,.0f} TL")
    print()

# Örnek
full_analysis("THYAO", portfolio_tl=100_000)



  THYAO — SIMONS KANTİTATİF ANALİZ
  Fiyat      : 324.50 TL
  1G         : -1.52%
  5G         : -2.70%
  20G        : -0.08%

── HMM Rejim Analizi ──────────────────────────
  Güncel rejim : Sideways
  Son 5 gün    : ['Bull', 'Bull', 'Bull', 'Sideways', 'Sideways']
  Bear      : %0.0
  Sideways  : %77.6
  Bull      : %22.4

── Edge Skorları ──────────────────────────────
  E1_momentum: ✗
  E2_squeeze: ✗
  E3_volume: ✓
  E4_rsi_momentum: ✗
  E5_ma_cross: ✗
  E6_rsc: ✗
  TOPLAM: 1/6

── Kelly Pozisyon ─────────────────────────────
  Geçmiş işlem : 61 adet
  Win rate     : %39.3
  Payoff oranı : 1.04x
  Tam Kelly    : %-18.98
  Quarter Kelly: %0.00
  Pozisyon TL  : 0 TL



In [17]:
# Analiz etmek istediğin hisseyi buraya yaz
HISSE = "GARAN"
PORTFOY_TL = 100_000

full_analysis(HISSE, portfolio_tl=PORTFOY_TL)



  GARAN — SIMONS KANTİTATİF ANALİZ
  Fiyat      : 125.70 TL
  1G         : -0.87%
  5G         : -0.79%
  20G        : -12.53%

── HMM Rejim Analizi ──────────────────────────
  Güncel rejim : Bear
  Son 5 gün    : ['Bear', 'Bear', 'Bear', 'Bear', 'Bear']
  Bear      : %99.9
  Sideways  : %0.1
  Bull      : %0.0

── Edge Skorları ──────────────────────────────
  E1_momentum: ✗
  E2_squeeze: ✗
  E3_volume: ✗
  E4_rsi_momentum: ✗
  E5_ma_cross: ✗
  E6_rsc: ✗
  TOPLAM: 0/6

── Kelly Pozisyon ─────────────────────────────
  Geçmiş işlem : 75 adet
  Win rate     : %48.0
  Payoff oranı : 1.22x
  Tam Kelly    : %5.36
  Quarter Kelly: %1.34
  Pozisyon TL  : 1,339 TL



In [18]:
# ── ÖĞRENEN SİSTEM 1: Sinyal Logger + Sonuç Değerlendirici ───────────────────
# Bugünkü sinyalleri kaydeder, geçmiş sinyallerin sonuçlarını değerlendirir.

import csv
from pathlib import Path

SIGNALS_CSV = Path(ROOT) / "raporlar" / "sinyal_gecmisi.csv"
HOLD_DAYS   = 10   # Kaç gün sonra sonucu değerlendir


def log_signals(filtered_results, price_data, market_regime):
    today = datetime.now().strftime("%Y-%m-%d")
    rows  = []
    for r in filtered_results:
        sym = r["hisse"]
        df  = price_data.get(sym)
        if df is None or df.empty:
            continue
        rows.append({
            "tarih":           today,
        "hisse":           sym,
            "fiyat_giris":     round(float(df["close"].iloc[-1]), 4),
            "benzerlik_skoru": r.get("benzerlik_skoru", 0),
            "birikim_skoru":   r.get("birikim_skoru", "0/5"),
            "tahmini_yukselis":r.get("tahmini_yukselis", 0),
            "market_regime":   market_regime,
            "cikis_tarihi":    "",
            "fiyat_cikis":     "",
            "gercek_getiri_%": "",
            "sonuc":           "bekliyor",
        })
    if not rows:
        print("Kaydedilecek sinyal yok.")
        return

    file_exists = SIGNALS_CSV.exists()
    with open(SIGNALS_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)
    print(f"{len(rows)} sinyal kaydedildi -> {SIGNALS_CSV}")


def evaluate_past_signals(price_data):
    if not SIGNALS_CSV.exists():
        print("Henuz kaydedilmis sinyal yok.")
        return pd.DataFrame()

    df = pd.read_csv(SIGNALS_CSV)
    bekleyen = df[df["sonuc"] == "bekliyor"].copy()
    if bekleyen.empty:
        print("Degerlendirme bekleyen sinyal yok.")
        return df

    today = datetime.now()
    updated = 0
    for idx, row in bekleyen.iterrows():
        try:
            giris_tarihi = datetime.strptime(row["tarih"], "%Y-%m-%d")
        except Exception:
            continue
        gun_farki = (today - giris_tarihi).days
        if gun_farki < HOLD_DAYS:
            continue

        sym = row["hisse"]
        pdata = price_data.get(sym)
        if pdata is None or pdata.empty:
            continue

        giris_fiyat = float(row["fiyat_giris"])
        son_fiyat   = float(pdata["close"].iloc[-1])
        getiri      = (son_fiyat - giris_fiyat) / (giris_fiyat + 1e-9)

        df.at[idx, "fiyat_cikis"]     = round(son_fiyat, 4)
        df.at[idx, "gercek_getiri_%"] = round(getiri * 100, 2)
        df.at[idx, "cikis_tarihi"]    = today.strftime("%Y-%m-%d")
        df.at[idx, "sonuc"]           = "kazanc" if getiri > 0 else "kayip"
        updated += 1

    df.to_csv(SIGNALS_CSV, index=False)
    print(f"{updated} sinyal guncellendi.")

    bitti = df[df["sonuc"].isin(["kazanc", "kayip"])].copy()
    if not bitti.empty:
        bitti["gercek_getiri_%"] = pd.to_numeric(bitti["gercek_getiri_%"], errors="coerce")
        win_rate = (bitti["sonuc"] == "kazanc").mean()
        avg_ret  = bitti["gercek_getiri_%"].mean()
        print(f"\nGecmis Performans ({len(bitti)} islem):")
        print(f"  Win Rate    : %{win_rate*100:.1f}")
        print(f"  Ort. Getiri : %{avg_ret:.2f}")
        print(f"  Toplam Getiri: %{bitti['gercek_getiri_%'].sum():.2f}")
        return bitti
    return df


# Bugünkü sinyalleri kaydet + geçmişi değerlendir
log_signals(FILTERED_RESULTS, PRICE_DATA, MARKET_REGIME)
print()
evaluate_past_signals(PRICE_DATA)


590 sinyal kaydedildi -> /content/drive/MyDrive/Simons_Quant/raporlar/sinyal_gecmisi.csv

585 sinyal guncellendi.

Gecmis Performans (585 islem):
  Win Rate    : %35.0
  Ort. Getiri : %-0.85
  Toplam Getiri: %-496.02


,tarih,hisse,fiyat_giris,benzerlik_skoru,birikim_skoru,tahmini_yukselis,market_regime,cikis_tarihi,fiyat_cikis,gercek_getiri_%,sonuc
0,2026-07-01,SASA,2.43,0.997,3/5,42.7,Unknown,2026-07-20,2.44,0.41,kazanc
1,2026-07-01,ONCSM,250.50,0.995,3/5,119.4,Unknown,2026-07-20,273.75,9.28,kazanc
2,2026-07-01,KUTPO,87.05,0.995,3/5,60.5,Unknown,2026-07-20,85.45,-1.84,kayip
3,2026-07-01,KIMMR,15.88,0.995,4/5,51.3,Unknown,2026-07-20,15.62,-1.64,kayip
4,2026-07-01,NATEN,6.46,0.994,3/5,79.7,Unknown,2026-07-20,6.34,-1.86,kayip
...,...,...,...,...,...,...,...,...,...,...,...
580,2026-07-01,GENIL,9.15,0.937,1/5,117.0,Unknown,2026-07-20,9.19,0.44,kazanc
581,2026-07-01,ZGYO,38.32,0.936,2/5,130.4,Unknown,2026-07-20,38.94,1.62,kazanc
582,2026-07-01,OYAYO,49.00,0.935,2/5,50.3,Unknown,2026-07-20,47.52,-3.02,kayip
583,2026-07-01,MMCAS,59.50,0.934,1/5,50.2,Unknown,2026-07-20,65.60,10.25,kazanc


In [19]:
# ── ÖĞRENEN SİSTEM 2: Otomatik Parametre Tuner ───────────────────────────────
# Geçmiş sinyal sonuçlarına bakarak MIN_SIMILARITY ve diğer eşikleri ayarlar.

PARAMS_CSV = Path(ROOT) / "raporlar" / "param_gecmisi.csv"


def auto_tune_parameters():
    global MIN_SIMILARITY, MIN_RISE_PCT, HOLD_DAYS

    if not SIGNALS_CSV.exists():
        print("Henuz yeterli veri yok (sinyal_gecmisi.csv bulunamadi).")
        return

    df = pd.read_csv(SIGNALS_CSV)
    df = df[df["sonuc"].isin(["kazanc", "kayip"])].copy()
    if len(df) < 10:
        print(f"Yeterli gecmis sinyal yok ({len(df)}/10 minimum). Parametreler degismedi.")
        return

    df["gercek_getiri_%"] = pd.to_numeric(df["gercek_getiri_%"], errors="coerce")
    df["benzerlik_skoru"] = pd.to_numeric(df["benzerlik_skoru"], errors="coerce")

    win_rate = (df["sonuc"] == "kazanc").mean()
    avg_ret  = df["gercek_getiri_%"].mean()

    print(f"Gecmis performans: win=%{win_rate*100:.1f} | ort_getiri=%{avg_ret:.2f} ({len(df)} islem)")

    old_sim = MIN_SIMILARITY
    old_rise = MIN_RISE_PCT

    # Benzerlik skoru bucketlarina gore win rate hesapla
    buckets = [
        (0.85, 1.01),
        (0.75, 0.85),
        (0.65, 0.75),
        (0.00, 0.65),
    ]
    best_wr   = 0.0
    best_thr  = MIN_SIMILARITY
    for lo, hi in buckets:
        sub = df[(df["benzerlik_skoru"] >= lo) & (df["benzerlik_skoru"] < hi)]
        if len(sub) < 5:
            continue
        wr = (sub["sonuc"] == "kazanc").mean()
        print(f"  Benzerlik [{lo:.2f}-{hi:.2f}): n={len(sub)}, win=%{wr*100:.1f}")
        if wr > best_wr:
            best_wr  = wr
            best_thr = lo

    # Yalnizca belirgin fark varsa guncelle
    if best_thr != MIN_SIMILARITY and best_wr >= 0.52:
        MIN_SIMILARITY = max(0.55, min(best_thr, 0.90))
        print(f"  MIN_SIMILARITY: {old_sim:.2f} -> {MIN_SIMILARITY:.2f}")
    else:
        print(f"  MIN_SIMILARITY: {old_sim:.2f} (degismedi)")

    # Genel win rate dusukse daha katı filtre
    if win_rate < 0.45:
        MIN_RISE_PCT = min(MIN_RISE_PCT + 0.05, 0.50)
        print(f"  WIN RATE DUSUK! MIN_RISE_PCT: {old_rise:.2f} -> {MIN_RISE_PCT:.2f}")
    elif win_rate > 0.60 and MIN_RISE_PCT > 0.25:
        MIN_RISE_PCT = max(MIN_RISE_PCT - 0.05, 0.20)
        print(f"  WIN RATE YüKSEK! MIN_RISE_PCT: {old_rise:.2f} -> {MIN_RISE_PCT:.2f}")
    else:
        print(f"  MIN_RISE_PCT: {old_rise:.2f} (degismedi)")

    # Parametre gecmisini kaydet
    param_row = {
        "tarih":           datetime.now().strftime("%Y-%m-%d %H:%M"),
        "n_islem":         len(df),
        "win_rate_%":      round(win_rate * 100, 1),
        "avg_ret_%":       round(avg_ret, 2),
        "MIN_SIMILARITY":  round(MIN_SIMILARITY, 3),
        "MIN_RISE_PCT":    round(MIN_RISE_PCT, 2),
        "HOLD_DAYS":       HOLD_DAYS,
    }
    file_exists = PARAMS_CSV.exists()
    with open(PARAMS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=param_row.keys())
        if not file_exists:
            w.writeheader()
        w.writerow(param_row)
    print(f"\nParametre gecmisi guncellendi -> {PARAMS_CSV}")
    print(f"Aktif parametreler: MIN_SIMILARITY={MIN_SIMILARITY:.3f} | MIN_RISE_PCT={MIN_RISE_PCT:.2f}")


auto_tune_parameters()


Gecmis performans: win=%35.0 | ort_getiri=%-0.85 (585 islem)
  Benzerlik [0.85-1.01): n=585, win=%35.0
  MIN_SIMILARITY: 0.65 (degismedi)
  WIN RATE DUSUK! MIN_RISE_PCT: 0.30 -> 0.35

Parametre gecmisi guncellendi -> /content/drive/MyDrive/Simons_Quant/raporlar/param_gecmisi.csv
Aktif parametreler: MIN_SIMILARITY=0.650 | MIN_RISE_PCT=0.35


In [20]:
# ── TEK TIKLA DEPLOY: full_pipeline() ────────────────────────────────────────
# Tum adimlar otomatik calisir:
#   1. Parametre tuning (gecmis veriye gore)
#   2. Veri guncelleme
#   3. HMM rejim tespiti
#   4. Pattern madenciligi + benzerlik taramasi
#   5. Filtre A+B uygulamasi
#   6. Edge tarama
#   7. Nihai rapor + Excel
#   8. Sinyal kaydi (gunluk log)

def full_pipeline(portfoy_tl: float = 100_000):
    import gc
    global PRICE_DATA, PATTERN_MATRIX, PATTERN_LABELS, PATTERN_SCALER
    global SIMILARITY_RESULTS, FILTERED_RESULTS, edge_results, pair_signals
    global MARKET_REGIME, BENCHMARK_CLOSE, BENCH

    start = time.time()
    now_str = datetime.now().strftime("%d.%m.%Y %H:%M")
    sep = "=" * 65
    print(f"\n{sep}")
    print(f"  SIMONS QUANT BIST -- TAM OTOMATIK PIPELINE -- {now_str}")
    print(f"{sep}\n")

    # ── ADIM 0: Parametre Tuning ─────────────────────────────────────────────
    print("[0/7] Parametre guncelleniyor...")
    auto_tune_parameters()

    # ── ADIM 1: Sembolleri al ────────────────────────────────────────────────
    print("\n[1/7] Sembol listesi...")
    symbols = get_bist_symbols()
    print(f"  {len(symbols)} hisse")

    # ── ADIM 2: Fiyat verisi ─────────────────────────────────────────────────
    print("\n[2/7] Fiyat verisi indiriliyor...")
    PRICE_DATA = {}
    failed_syms = []
    for sym in tqdm(symbols, desc="Veri"):
        df = get_price(sym)
        if not df.empty:
            PRICE_DATA[sym] = df
        else:
            failed_syms.append(sym)
        time.sleep(0.08)
    print(f"  Basarili: {len(PRICE_DATA)} | Basarisiz: {len(failed_syms)}")

    # ── ADIM 3: HMM rejim ────────────────────────────────────────────────────
    print("\n[3/7] Piyasa rejimi...")
    for attempt in range(3):
        try:
            _b = TV.get_hist("XU100", "TVC", Interval.in_daily, n_bars=700)
            if _b is not None and len(_b) > 100:
                _b.columns = [c.lower() for c in _b.columns]
                _b.index = pd.to_datetime(_b.index)
                BENCH = _b
                BENCHMARK_CLOSE = _b["close"]
                break
        except Exception:
            time.sleep(2 ** attempt)
    if not BENCH.empty:
        hmm_r = fit_hmm(BENCH["close"])
        MARKET_REGIME = hmm_r["regime"]
    else:
        MARKET_REGIME = "Unknown"
    print(f"  Rejim: {MARKET_REGIME}")

    # ── ADIM 4: Pattern madenciligi + benzerlik ───────────────────────────────
    print("\n[4/7] Pattern madenciligi...")
    PATTERN_MATRIX, PATTERN_LABELS, PATTERN_SCALER = mine_patterns(PRICE_DATA)
    print(f"  {len(PATTERN_LABELS)} pattern bulundu")

    print("\n[5/7] Benzerlik taramasi...")
    SIMILARITY_RESULTS = scan_by_similarity(
        PRICE_DATA, PATTERN_MATRIX, PATTERN_LABELS, PATTERN_SCALER
    )
    print(f"  {len(SIMILARITY_RESULTS)} hisse benzer pattern tasiyor")

    # ── ADIM 5: Filtre A+B ───────────────────────────────────────────────────
    print("\n[6/7] Filtreler (A+B) uygulanıyor...")
    FILTERED_RESULTS = []
    f_stats = {"tamamlandi": 0, "gecti": 0}
    for r in SIMILARITY_RESULTS:
        sym = r["hisse"]
        df  = PRICE_DATA.get(sym)
        if df is None or df.empty:
            continue
        close  = df["close"]
        volume = df["volume"] if "volume" in df.columns else pd.Series(np.ones(len(close)), index=close.index)
        volume = volume.fillna(1).replace(0, 1)
        fa = is_rise_complete(close, volume)
        if fa["complete"]:
            f_stats["tamamlandi"] += 1
            continue
        fb = detect_accumulation(close, volume)
        FILTERED_RESULTS.append({
            **r,
            "yukselis_40d_%":  fa["rise_40d_pct"],
            "momentum":        fa["momentum"],
            "birikim_skoru":   f"{fb['score']}/5",
            "birikim":         "EVET" if fb["accumulating"] else "hayir",
            "band_%":          fb["band_pct"],
            "obv":             fb["obv_slope"],
            "smart_money":     fb["smart_money"],
            "erken_mi":        fb["erken_mi"],
        })
        f_stats["gecti"] += 1
    FILTERED_RESULTS.sort(key=lambda x: (1 if x["birikim"] == "EVET" else 0, x["benzerlik_skoru"]), reverse=True)
    print(f"  Elenen: {f_stats['tamamlandi']} | Gecen: {f_stats['gecti']}")

    # ── ADIM 6: Edge tarama ──────────────────────────────────────────────────
    print("\n[7/7] Edge taramasi...")
    edge_results = []
    for sym, df in tqdm(PRICE_DATA.items(), desc="Edge"):
        try:
            close  = df["close"]
            volume = df.get("volume", pd.Series(np.ones(len(close)), index=close.index))
            volume = volume.fillna(1).replace(0, 1)
            sc = calc_edge_score(close, volume)
            if sc["score"] < 2:
                continue
            hmm_s = fit_hmm(close, n_states=2)
            delta = close.diff()
            gain  = delta.where(delta>0, 0).rolling(14, min_periods=5).mean()
            loss  = (-delta).where(-delta>0, 0).rolling(14, min_periods=5).mean()
            rsi_s = 100 - 100 / (1 + gain / (loss + 1e-9))
            obv_s = (np.sign(close.diff()) * volume).cumsum()
            obv_slope_s = obv_s.rolling(10).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True)
            sig_s = (rsi_s < 60) & (obv_slope_s > 0)
            bt    = backtest_win_rate(close.iloc[-252:], sig_s.iloc[-252:], hold_days=5)
            kelly = kelly_position(bt["win_rate"], bt["avg_win"], bt["avg_loss"])
            edge_results.append({
                "hisse":    sym,
                "fiyat":    round(float(close.iloc[-1]), 2),
                "skor":     f"{sc['score']}/{sc['max_score']}",
                "win_%":    round(bt["win_rate"] * 100, 1),
                "kelly_%":  kelly["kelly_frac_pct"],
            })
        except Exception:
            continue
    print(f"  {len(edge_results)} edge sinyali")

    # ── Rapor ────────────────────────────────────────────────────────────────
    ts_s = datetime.now().strftime("%Y%m%d_%H%M")
    xl_s = f"{ROOT}/raporlar/simons_quant_{ts_s}.xlsx"
    with pd.ExcelWriter(xl_s, engine="openpyxl") as writer:
        if FILTERED_RESULTS:
            pd.DataFrame(FILTERED_RESULTS).to_excel(writer, sheet_name="Filtreli_Adaylar", index=False)
        if SIMILARITY_RESULTS:
            pd.DataFrame(SIMILARITY_RESULTS).to_excel(writer, sheet_name="Pattern_Benzerligi", index=False)
        if edge_results:
            pd.DataFrame(edge_results).to_excel(writer, sheet_name="Edge_Sinyaller", index=False)
        if SIGNALS_CSV.exists():
            pd.read_csv(SIGNALS_CSV).to_excel(writer, sheet_name="Sinyal_Gecmisi", index=False)
        if PARAMS_CSV.exists():
            pd.read_csv(PARAMS_CSV).to_excel(writer, sheet_name="Param_Gecmisi", index=False)

    # ── Sinyal kaydi ─────────────────────────────────────────────────────────
    log_signals(FILTERED_RESULTS, PRICE_DATA, MARKET_REGIME)
    elapsed = time.time() - start

    print(f"\n{sep}")
    print(f"  TAMAMLANDI -- {elapsed/60:.1f} dakika")
    print(f"  Taranan hisse       : {len(PRICE_DATA)}")
    print(f"  Piyasa rejimi       : {MARKET_REGIME}")
    print(f"  Filtreli adaylar    : {len(FILTERED_RESULTS)}")
    print(f"  Edge sinyalleri     : {len(edge_results)}")
    print(f"  MIN_SIMILARITY      : {MIN_SIMILARITY:.3f}")
    print(f"  MIN_RISE_PCT        : {MIN_RISE_PCT:.2f}")
    print(f"  Excel               : {xl_s}")
    print(f"{sep}")

    if FILTERED_RESULTS:
        df_f = pd.DataFrame(FILTERED_RESULTS)
        print("\n── EN IYI 10 ADAY ──────────────────────────────────────")
        cols = ["hisse", "fiyat", "benzerlik_skoru", "birikim", "birikim_skoru", "tahmini_yukselis"]
        print(df_f[cols].head(10).to_string(index=False))

    gc.collect()


# -- Calistirmak icin: -------------------------------------------------------
# full_pipeline(portfoy_tl=100_000)
print("full_pipeline() hazir. Calistirmak icin son satirdaki yorum kaldir.")
print("ya da dogrudan calistir:")
print("  full_pipeline(portfoy_tl=100_000)")


full_pipeline() hazir. Calistirmak icin son satirdaki yorum kaldir.
ya da dogrudan calistir:
  full_pipeline(portfoy_tl=100_000)
